<a href="https://colab.research.google.com/github/koki-shiroyama0430/Complete-Data-Science-Bootcamp---Udemy/blob/main/03_MNIST_Handwritten_Digit_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds

### Loading Dataset



In [ ]:
# Load MNIST: with_info for metadata, as_supervised for (input, label) tuples
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

# Split into train and test sets
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

# Define number of validation samples (10% of training data)
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

# Define number of test samples
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

# Scale pixel values to [0,1] range for faster convergence
def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255.
    return image, label

scaled_train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

# Shuffle and split into Train and Validation
BUFFER_SIZE = 10000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

# Set batch size for mini-batch gradient descent
BATCH_SIZE = 100

# Batching datasets to update weights every 100 samples
train_data = train_data.batch(BATCH_SIZE)

# Batching validation/test data as a single block for faster evaluation
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

# Extract one batch from the validation pipeline for evaluation
validation_inputs, validation_targets = next(iter(validation_data))

### Model Configuration
Define the basic dimensions of neural network.
Use a Sequential model to stack layers linearly.

In [ ]:
# Hyperparameters: Defining network dimensions
input_size = 784     # 28x28 pixels
output_size = 10    # Digits 0-9
hidden_layer_size = 50 # Number of neurons in each hidden layer

# Model Architecture: Defining the feed-forward structure
model = tf.keras.Sequential([
    # Convert 2D image (28x28) into 1D vector (784)
    tf.keras.layers.Flatten(input_shape=(28,28,1)),

    # Hidden layers with ReLU to capture non-linear patterns
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    # Output layer with Softmax to output class probabilities
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# Display the model's architecture and total trainable parameters
model.summary()

### Compile
Define how the model should learn.

In [ ]:
# Configure the learning process
model.compile(
    optimizer='adam',  # Adaptive moment Estimation
    loss='sparse_categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']  # Monitor the percentage of correct predictions
)

### Model Training

In [ ]:
# Set the number of training iterations
NUM_EPOCHS = 5

# Start the training process
# The model learns from train_data while monitoring performance on validation_data
model.fit(
    train_data,
    epochs=NUM_EPOCHS,
    validation_data=(validation_inputs, validation_targets),
    verbose=2
)

### Model Evaluation

In [ ]:
# Evaluate the final model performance using the test dataset
test_loss, test_accuracy = model.evaluate(test_data)

# Print the final results in a clean format
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

### Save Model

In [ ]:
# Save the trained model in Keras format
# This 'keras' file is what you will later upload to Vertex AI
model.save('mnist_model.keras')